#### Unit Testing – Bronze Menu Items
This notebook validates the Bronze ingestion of menu_items data by
comparing the ingested Bronze table with the raw source files stored in
Databricks Volumes.

The tests focus on:
- Row count consistency
- Aggregate value validation
- Null checks
- Duplicate detection



In [0]:
-- Row count comparison between raw volume and bronze table
SELECT
  'bronze_table' AS source_type,
  COUNT(*) AS row_count
FROM coffee.bronze.menu_items

UNION ALL

SELECT
  'raw_volume' AS source_type,
  COUNT(*) AS row_count
FROM (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/default/coffee_raw_volume/menu_items/',
    format => 'csv',
    header => true
  )
);


In [0]:
-- Total price comparison between raw volume and bronze table
SELECT
  'bronze_table' AS source_type,
  SUM(CAST(price AS DOUBLE)) AS total_price
FROM coffee.bronze.menu_items

UNION ALL

SELECT
  'raw_volume' AS source_type,
  SUM(CAST(price AS DOUBLE)) AS total_price
FROM (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/default/coffee_raw_volume/menu_items',
    format => 'csv',
    header => true
  )
);


In [0]:

-- null checks 

SELECT
  SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS null_item_id_count,
  SUM(CASE WHEN item_name IS NULL THEN 1 ELSE 0 END) AS null_item_name_count
FROM coffee.bronze.menu_items;


In [0]:
--duplicate check 
SELECT
 item_id,
 item_name,
  COUNT(*) AS cnt
FROM coffee.bronze.menu_items
GROUP BY 1,2
HAVING COUNT(*) > 1;
